In [21]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None

import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import os


In [22]:
train_path = "NSLKDD_train.csv"
test_path  = "NSLKDD_test.csv"

train_df = pd.read_csv(train_path, header=None)
test_df  = pd.read_csv(test_path,  header=None)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print(train_df.head(3))


Train shape: (118813, 42)
Test shape : (29705, 42)
         0              1        2     3          4          5     6   \
0  duration  protocol_type  service  flag  src_bytes  dst_bytes  land   
1         0              1       30     5          0          0     0   
2         0              1       69     5          0          0     0   

               7       8    9   ...                  32  \
0  wrong_fragment  urgent  hot  ...  dst_host_srv_count   
1               0       0    0  ...                  18   
2               0       0    0  ...                  19   

                       33                      34  \
0  dst_host_same_srv_rate  dst_host_diff_srv_rate   
1                    0.07                    0.07   
2                    0.07                    0.07   

                            35                           36  \
0  dst_host_same_src_port_rate  dst_host_srv_diff_host_rate   
1                          0.0                          0.0   
2                

In [23]:
base_features = [
 'duration','protocol_type','service','flag','src_bytes','dst_bytes','land',
 'wrong_fragment','urgent','hot','num_failed_logins','logged_in',
 'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
 'num_shells','num_access_files','num_outbound_cmds','is_host_login',
 'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
 'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
 'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
 'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
 'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
 'dst_host_rerror_rate','dst_host_srv_rerror_rate'
]

train_df.columns = base_features + ["label"]
test_df.columns  = base_features + ["label"]

print(train_df[["dst_host_srv_diff_host_rate","dst_host_srv_rerror_rate","label"]].head())


   dst_host_srv_diff_host_rate  dst_host_srv_rerror_rate   label
0  dst_host_srv_diff_host_rate  dst_host_srv_rerror_rate  labels
1                          0.0                       0.0      14
2                          0.0                       0.0      14
3                         0.18                       1.0      16
4                          0.0                       0.0      14


In [24]:
# Convert label to numeric; 'labels' will become NaN
train_df["label_num"] = pd.to_numeric(train_df["label"], errors="coerce")
test_df["label_num"]  = pd.to_numeric(test_df["label"],  errors="coerce")

print("Unique before drop:", train_df["label"].unique()[:20])

# Drop header row(s) where conversion failed
train_df = train_df.dropna(subset=["label_num"]).reset_index(drop=True)
test_df  = test_df.dropna(subset=["label_num"]).reset_index(drop=True)

train_df["label_num"] = train_df["label_num"].astype(int)
test_df["label_num"]  = test_df["label_num"].astype(int)

print("\nUnique numeric labels (train):", np.sort(train_df["label_num"].unique()))
print("Unique numeric labels (test):", np.sort(test_df["label_num"].unique()))


Unique before drop: ['labels' '14' '16' '4' '35' '7' '1' '25' '11' '34' '15' '0' '21' '20'
 '27' '29' '32' '28' '10' '19']

Unique numeric labels (train): [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39]
Unique numeric labels (test): [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 32 34 35 37 38 39]


In [11]:
dos_attacks = [
 "back","land","neptune","pod","smurf","teardrop",
 "mailbomb","processtable","udpstorm","apache2","worm"
]

probe_attacks = [
 "satan","ipsweep","nmap","portsweep","mscan","saint"
]

r2l_attacks = [
 "guess_passwd","ftp_write","imap","phf","multihop","warezmaster","warezclient",
 "spy","xlock","xsnoop","snmpguess","snmpgetattack","httptunnel","sendmail","named"
]

u2r_attacks = [
 "buffer_overflow","loadmodule","rootkit","perl","sqlattack","xterm","ps"
]

def map_to_5class(label):
    l = str(label).strip().lower().replace(".", "")
    if l == "normal":
        return "normal"
    if l in dos_attacks:
        return "dos"
    if l in probe_attacks:
        return "probe"
    if l in r2l_attacks:
        return "r2l"
    if l in u2r_attacks:
        return "u2r"
    return None   # unknown / weird label

train_df["target"] = train_df["label"].apply(map_to_5class)
test_df["target"]  = test_df["label"].apply(map_to_5class)

print("Train class counts:\n", train_df["target"].value_counts(), "\n")
print("Test class counts:\n",  test_df["target"].value_counts())


Train class counts:
 Series([], Name: count, dtype: int64) 

Test class counts:
 Series([], Name: count, dtype: int64)


In [12]:
# Drop rows with unknown mapping (just in case)
train_df = train_df.dropna(subset=["target"]).reset_index(drop=True)
test_df  = test_df.dropna(subset=["target"]).reset_index(drop=True)

# Remove difficulty if present
drop_cols = ["label", "target"]
if "difficulty" in train_df.columns:
    drop_cols.append("difficulty")

X_train = train_df.drop(columns=drop_cols)
X_test  = test_df.drop(columns=drop_cols)

classes = ["dos", "normal", "probe", "r2l", "u2r"]
class_to_id = {c: i for i, c in enumerate(classes)}

y_train = train_df["target"].map(class_to_id).astype(int).values
y_test  = test_df["target"].map(class_to_id).astype(int).values

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train classes:", np.unique(y_train), "->", classes)


X_train: (0, 41) X_test: (0, 41)
y_train classes: [] -> ['dos', 'normal', 'probe', 'r2l', 'u2r']


In [ ]:
cat_cols = ["protocol_type", "service", "flag"]

train_cat = pd.get_dummies(X_train[cat_cols], prefix=cat_cols).astype(int)
test_cat  = pd.get_dummies(X_test[cat_cols],  prefix=cat_cols).astype(int)

# Align categorical columns between train & test
all_cat_cols = train_cat.columns.union(test_cat.columns)
train_cat = train_cat.reindex(columns=all_cat_cols, fill_value=0)
test_cat  = test_cat.reindex(columns=all_cat_cols, fill_value=0)

# Numeric part = everything except these categoricals
num_train = X_train.drop(columns=cat_cols).astype(float)
num_test  = X_test.drop(columns=cat_cols).astype(float)

# Combine numeric + cat
X_train_full = pd.concat([num_train.reset_index(drop=True),
                          train_cat.reset_index(drop=True)], axis=1)

X_test_full  = pd.concat([num_test.reset_index(drop=True),
                          test_cat.reset_index(drop=True)], axis=1)

print("Final feature count:", X_train_full.shape[1])


Saved: nslkdd_multi_scaler.pkl and nslkdd_multi_features.pkl


In [13]:
models = {
    "decision_tree": DecisionTreeClassifier(max_depth=25),
    "random_forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "extra_trees":   ExtraTreesClassifier(n_estimators=300, random_state=42),
    "naive_bayes":   GaussianNB(),
    "logistic_regression": LogisticRegression(max_iter=2000, multi_class="multinomial")
}

if LGBMClassifier is not None:
    models["lightgbm"] = LGBMClassifier(
        n_estimators=400,
        num_leaves=64,
        verbose=-1
    )

results = []

for name, model in models.items():
    print(f"Training multiclass model: {name}")
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted")
    rec  = recall_score(y_test, y_pred, average="weighted")
    f1   = f1_score(y_test, y_pred, average="weighted")

    results.append((name, acc, prec, rec, f1))

    save_path = f"models/nslkdd_multi_{name}.pkl"
    joblib.dump(model, save_path)
    print(f"  Saved → {save_path} | Acc={acc:.4f}")

results_df = pd.DataFrame(results, columns=["Model","Accuracy","Precision","Recall","F1"])
results_df


Training multiclass model: decision_tree


ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required by DecisionTreeClassifier.

In [ ]:
best = results_df.iloc[results_df["Accuracy"].idxmax()]
print("Best:", best["Model"], best["Accuracy"])

best_model = models[best["Model"]]
y_pred_best = best_model.predict(X_test_scaled)

cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=classes, yticklabels=classes,
            cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Confusion Matrix - {best['Model']}")
plt.show()

print(classification_report(y_test, y_pred_best, target_names=classes))
